In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\pulib\AppData\Local\Temp\ipykernel_11968\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def load_all_pdf(file_path):
    """
    Load all PDF files from the specified file path.
    """
    file_path = Path(file_path)

    pdf_files = list(file_path.glob("*.pdf"))
    print(pdf_files)
    all_documents = []

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = load_all_pdf("../data/pdf")


[WindowsPath('../data/pdf/attention-is-all-you-need-Paper.pdf'), WindowsPath('../data/pdf/Computer_Vision_and_Image_Processing_A_Paper_Revie.pdf'), WindowsPath('../data/pdf/rag_llm.pdf')]

Processing: attention-is-all-you-need-Paper.pdf
✓ Loaded 11 pages

Processing: Computer_Vision_and_Image_Processing_A_Paper_Revie.pdf
✓ Loaded 9 pages

Processing: rag_llm.pdf
✓ Loaded 21 pages

Total documents loaded: 41


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

In [4]:
# pdf_file = Path("../data/python.pdf")

# creates a Path object.

# That object has many built-in properties and methods:

# pdf_file.name
# pdf_file.stem
# pdf_file.suffix
# pdf_file.parent
# pdf_file.exists()
# pdf_file.is_file()

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

With overlap

Suppose overlap = 200

Chunk 1

Artificial Intelligence

Machine learning is...

-----------------------

Chunk 2

Machine learning is...

Supervised learning...

Decision trees...

The last part of Chunk 1 appears again at the beginning of Chunk 2.

This preserves context.


Why 200?

It's a balance.

Too small

↓

Context lost

Too large

↓

Duplicate data

More embeddings

More storage

100–200 is a common choice.

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 41 documents into 233 chunks

Example chunk:
Content: Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz...
Metadata: {'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiori

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 


###  embedding And vectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
import uuid
from typing import List, Dict, Any , Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [8]:
class EmbeddingManager:
    
        def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
            self.model_name = model_name
            self.model = None
            self._load_model()
            
        def _load_model(self):
            """Load the embedding model."""
            try: 
                self.model = SentenceTransformer(self.model_name)
                print(f"✓ Loaded embedding model: {self.model_name}")
            except Exception as e:
                print(f"✗ Error loading model {self.model_name}: {e}") 
        
        def generate_embeddings(self,texts: List[str]) -> np.ndarray:
            """Generate embeddings for a list of texts."""
            if not self.model:
                raise ValueError("Model is not loaded.")
            try:
                embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
                print(f"✓ Generated embeddings for {len(texts)} texts")
                print(f"Generated embeddings with shape: {embeddings.shape}")
                return embeddings
            except Exception as e:
                print(f"✗ Error generating embeddings: {e}")
                return np.array([])
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3251.18it/s]


✓ Loaded embedding model: all-MiniLM-L6-v2


#### vectorbase

In [9]:
import os 

In [10]:
import chromadb
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create a persistent vector database
client = chromadb.PersistentClient(path="./my_db")

# Create or open a collection
collection = client.get_or_create_collection(name="documents")

# Your text
texts = [
    "Python is easy to learn.",
    "Artificial Intelligence is fascinating."
]

# Generate embeddings
embeddings = model.encode(texts, convert_to_numpy=True)

# Store them
collection.add(
    ids=["1", "2"],
    documents=texts,
    embeddings=embeddings.tolist()
)

print("Done!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4775.90it/s]


Done!


In [11]:
class VectorDatabase:
    
    def __init__(self, collection_name:str = "documents", persistent_path:str = "./data/my_db"):
        
        self.collection_name = collection_name
        self.persistent_path = persistent_path
        self.client = None
        self.collection = None
        self._initialize_database()
        
    def _initialize_database(self):
        """Initialize the ChromaDB client and collection."""
        try:
            
            os.makedirs(self.persistent_path, exist_ok=True)
            
            # create a persistent client
            self.client = chromadb.PersistentClient(path=self.persistent_path)
            
            # get or create the collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name)
            print(f"✓ Initialized database at {self.persistent_path} with collection '{self.collection_name}'")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"✗ Error initializing database: {e}")
            
    def add_documents(self, documents: List[str], embeddings: np.ndarray):
        
        
        # Validate input lengths
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        
        print(f"Adding {len(documents)} documents to the collection '{self.collection_name}'")
        
        # prepare data for chromadb 
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        # IDs
        # Texts
        # Metadata
        # Embeddings
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)): # enumerate() -> gives index values to the zip() pairs
            # documents = [
            #     chunk1,
            #     chunk2,
            #     chunk3
            # ]

            # embeddings = [
            #     vector1,
            #     vector2,
            #     vector3
            # ]
            
            
            # for i = 2 then uuid gives doc_8f2c2f00_2
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" 
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata) # With dict() python creates a copy of the metadata dictionary. This is important to avoid modifying the original metadata of the document.
            # this how the copy looks like 
            # {
            # "source":"AI.pdf",
            # "page":5
            # }
            
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())   
            
                # embedding is NumPy array.
                # array([0.12,0.43,0.91])
                # ChromaDB wants list of floats, so we convert it to list using .tolist()
                # [0.12,0.43,0.91] 
        
            # Add to collection
        try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )
                print(f"Successfully added {len(documents)} documents to vector store")
                print(f"Total documents in collection: {self.collection.count()}")
                
        except Exception as e:
                print(f"Error adding documents to vector store: {e}")
                raise

vectorstore=VectorDatabase()
vectorstore
    
    

✓ Initialized database at ./data/my_db with collection 'documents'
Existing documents in collection: 233


In [12]:
# class VectorStore:
#     """Manages document embeddings in a ChromaDB vector store"""
    
#     def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
#         """
#         Initialize the vector store
        
#         Args:
#             collection_name: Name of the ChromaDB collection
#             persist_directory: Directory to persist the vector store
#         """
#         self.collection_name = collection_name
#         self.persist_directory = persist_directory
#         self.client = None
#         self.collection = None
#         self._initialize_store()

#     def _initialize_store(self):
#         """Initialize ChromaDB client and collection"""
#         try:
#             # Create persistent ChromaDB client
#             os.makedirs(self.persist_directory, exist_ok=True)
#             self.client = chromadb.PersistentClient(path=self.persist_directory)
            
#             # Get or create collection
#             self.collection = self.client.get_or_create_collection(
#                 name=self.collection_name,
#                 metadata={"description": "PDF document embeddings for RAG"}
#             )
#             print(f"Vector store initialized. Collection: {self.collection_name}")
#             print(f"Existing documents in collection: {self.collection.count()}")
            
#         except Exception as e:
#             print(f"Error initializing vector store: {e}")
#             raise

#     def add_documents(self, documents: List[Any], embeddings: np.ndarray):
#         """
#         Add documents and their embeddings to the vector store
        
#         Args:
#             documents: List of LangChain documents
#             embeddings: Corresponding embeddings for the documents
#         """
#         if len(documents) != len(embeddings):
#             raise ValueError("Number of documents must match number of embeddings")
        
#         print(f"Adding {len(documents)} documents to vector store...")
        
#         # Prepare data for ChromaDB
#         ids = []
#         metadatas = []
#         documents_text = []
#         embeddings_list = []
        
#         for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
#             # Generate unique ID
#             doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
#             ids.append(doc_id)
            
#             # Prepare metadata
#             metadata = dict(doc.metadata)
#             metadata['doc_index'] = i
#             metadata['content_length'] = len(doc.page_content)
#             metadatas.append(metadata)
            
#             # Document content
#             documents_text.append(doc.page_content)
            
#             # Embedding
#             embeddings_list.append(embedding.tolist())
        
#         # Add to collection
#         try:
#             self.collection.add(
#                 ids=ids,
#                 embeddings=embeddings_list,
#                 metadatas=metadatas,
#                 documents=documents_text
#             )
#             print(f"Successfully added {len(documents)} documents to vector store")
#             print(f"Total documents in collection: {self.collection.count()}")
            
#         except Exception as e:
#             print(f"Error adding documents to vector store: {e}")
#             raise

# vectorstore=VectorStore()
# vectorstore
    

In [13]:
chunks

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

In [14]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector database
vectorstore.add_documents(chunks,embeddings)

Batches: 100%|██████████| 8/8 [00:06<00:00,  1.16it/s]


✓ Generated embeddings for 233 texts
Generated embeddings with shape: (233, 384)
Adding 233 documents to the collection 'documents'
Successfully added 233 documents to vector store
Total documents in collection: 466


### rag retrieval 

In [21]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: vectorstore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
        #    results = {
        #     "ids":[
        #         ["1","3"]
        #     ],

        #     "documents":[
        #         [
        #             "Python is easy",
        #             "Python supports AI"
        #         ]
        #     ],

        #     "metadatas":[
        #         [
        #             {"page":1},
        #             {"page":5}
        #         ]
        #     ],

        #     "distances":[
        #         [
        #             0.08,
        #             0.15
        #         ]
        #     ]
        # }
            
            # Process results
            retrieved_docs = []
            # making sure the results contain documents and that the first element is not empty
            
            if results['documents'] and results['documents'][0]:
                # the reason for using [0] is that the results are structured as lists of lists, where each inner list corresponds to the results for a single query. Since we are only querying with one query, we access the first (and only) inner list to get the actual documents, metadatas, distances, and ids.
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [22]:
rag_retriever.retrieve("what is attention mechanism in deep learning", top_k=3, score_threshold=0.1)

Retrieving documents for query: 'what is attention mechanism in deep learning'
Top K: 3, Score threshold: 0.1


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.65it/s]

✓ Generated embeddings for 1 texts
Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_f397c328_25',
  'content': 'convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions\nfrom our models and present and discuss examples in the appendix. Not only do individual attention\nheads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic\nand semantic structure of the sentences.\n5 Training\nThis section describes the training regime for our models.\n5.1 Training Data and Batching\nWe trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million\nsentence pairs. Sentences were encoded using byte-pair encoding [ 3], which has a shared source-\ntarget vocabulary of about 37000 tokens. For English-French, we used the signiﬁcantly larger WMT\n2014 English-French dataset consisting of 36M sentences and split tokens 

In [23]:

rag_retriever.retrieve("what is computer vision", top_k=3, score_threshold=0.1)

Retrieving documents for query: 'what is computer vision'
Top K: 3, Score threshold: 0.1


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.09it/s]

✓ Generated embeddings for 1 texts
Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_e627f401_44',
  'content': 'process of obtaining information on events or descr iptions, from input scenes (digital images) and \nfeature extraction. The methods used to solve problems in computer vision depend on the \napplication domain and the nature of the data being analyzed. \nComputer vision is a combination of image processing and pattern recognition[2],[3]. The output \nof the Computer Vision process is image understanding. Development of this field is done  by \nadapting the ability of human vision in taking information. Computer Vision is the discipline of \nextracting information from i mages, as opposed to Computer Graphics [4]. The development of \ncomputer vision depends on the computer technology system, whether about image quality \nimprovement or image recognition. There is an overlap with Image Processing on basic techniques, \nand some authors use both terms interchangeably [4],[5].  \nThe primary purpose of Computer Vision is to create models and data 